In [58]:
import os
import time
import copy

import numpy       as     np
import matplotlib.pyplot as plt
from   PIL         import Image

import torch
import torch.nn    as     nn
from   torch.utils.data.sampler import SubsetRandomSampler
import torch.optim as optim
from torch.optim import lr_scheduler

import torchvision
from   torchvision import datasets, transforms

In [28]:
# load pre-trained ShuffleNetV2 ~8MB model
model = torch.hub.load('pytorch/vision', 'shufflenet_v2_x1_0', pretrained=True)

# can be vocal or noise
num_classes = 2

# replace fully-connected layer to match our dataset
# added 512 and dropout
model.fc    = nn.Sequential(nn.Linear(1024, 512),
                            nn.ReLU(inplace=True),
                            nn.Dropout(0.5),
                            nn.Linear(512, num_classes))

# use GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# transform input images to match network
trans = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Using cache found in /Users/gustavo/.cache/torch/hub/pytorch_vision_master


In [65]:
# prepare dataset
dataset_dir = "/Users/gustavo/Documents/git/vocalpy/training/dataset/vocal_noise"
train_dir = os.path.join(dataset_dir, "train")

# -- create training dataset1p
train_dataset = datasets.ImageFolder(train_dir, transform=trans)
class_names   = train_dataset.classes
num_classes   = len(class_names)
train_size    = len(train_dataset)
train_indices = list(range(train_size))
np.random.shuffle(train_indices)

print('train dataset has {} images'.format(train_size))
print('train dataset has {} classes:'.format(num_classes))
print(class_names)

# -- create testing dataset
test_dir      = os.path.join(dataset_dir, "test")
test_dataset  = datasets.ImageFolder(test_dir, transform=trans)
test_size     = len(test_dataset)
test_indices  = list(range(test_size))
np.random.shuffle(test_indices)

class_names   = test_dataset.classes
num_classes   = len(class_names)
train_size    = len(test_dataset)
print('test dataset has {} images'.format(train_size))
print('test dataset has {} classes:'.format(num_classes))
print(class_names)

# -- create dataloaders
train_sampler = SubsetRandomSampler(train_indices)
test_sampler  = SubsetRandomSampler(test_indices)

batch_size  = 128
dataloaders = {
    'train': torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, num_workers=4, sampler=train_sampler),
     'test': torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, num_workers=4, sampler=test_sampler)
}


train dataset has 20599 images
train dataset has 2 classes:
['noise', 'vocal']
test dataset has 2348 images
test dataset has 2 classes:
['noise', 'vocal']


In [36]:
# constant for classes
classes = ('noise', 'vocal')

# helper function to show an image
# (used in the `plot_classes_preds` function below)
def matplotlib_imshow(img, one_channel=False):
    if one_channel:
        img = img.mean(dim=0)
#     img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    if one_channel:
        plt.imshow(npimg, cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))

In [53]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [44]:
from torch.utils.tensorboard import SummaryWriter

# default `log_dir` is "runs" - we'll be more specific here
writer = SummaryWriter('./tb_exp')

In [43]:
# # get some random training images
# dataiter = iter(dataloaders['train'])
# images, labels = dataiter.next()

# # create grid of images
# img_grid = torchvision.utils.make_grid(images)

# # show images
# matplotlib_imshow(img_grid, one_channel=True)

# # write to tensorboard
# writer.add_image('sample training images', img_grid, 0)

# for n_iter in range(10000000000000000):
#     writer.add_scalar('Loss/train', np.random.random(), n_iter)
#     writer.add_scalar('Loss/test', np.random.random(), n_iter)
#     writer.add_scalar('Accuracy/train', np.random.random(), n_iter)
#     writer.add_scalar('Accuracy/test', np.random.random(), n_iter)
# writer.close()

In [63]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    aaa = 0
    for epoch in range(num_epochs):
        writer.add_scalar('epoch', epoch, epoch)
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                writer.add_scalar('running_loss', running_loss, aaa)
                writer.add_scalar('running_corrects', running_corrects, aaa)
                aaa = aaa + 1
                
            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            writer.add_scalar('epoch_loss', epoch_loss, epoch)
            writer.add_scalar('epoch_acc', epoch_acc, epoch)
            
            print('{} Loss: {:.4f} Acc: {:.4f}'.format(
                phase, epoch_loss, epoch_acc))

            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:4f}'.format(best_acc))

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model

In [46]:
def visualize_model(model, num_images=6):
    was_training = model.training
    model.eval()
    images_so_far = 0
    fig = plt.figure()

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(dataloaders['val']):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for j in range(inputs.size()[0]):
                images_so_far += 1
                ax = plt.subplot(num_images//2, 2, images_so_far)
                ax.axis('off')
                ax.set_title('predicted: {}'.format(class_names[preds[j]]))
                imshow(inputs.cpu().data[j])

                if images_so_far == num_images:
                    model.train(mode=was_training)
                    return
        model.train(mode=was_training)

In [64]:
model_ft = train_model(model, criterion, optimizer, exp_lr_scheduler, num_epochs=25)

Epoch 0/24
----------


Traceback (most recent call last):
  File "/Users/gustavo/anaconda3/lib/python3.7/multiprocessing/queues.py", line 242, in _feed
    send_bytes(obj)
  File "/Users/gustavo/anaconda3/lib/python3.7/multiprocessing/connection.py", line 200, in send_bytes
    self._send_bytes(m[offset:offset + size])
  File "/Users/gustavo/anaconda3/lib/python3.7/multiprocessing/connection.py", line 404, in _send_bytes
    self._send(header + buf)
  File "/Users/gustavo/anaconda3/lib/python3.7/multiprocessing/connection.py", line 368, in _send
    n = write(self._handle, buf)
BrokenPipeError: [Errno 32] Broken pipe
Traceback (most recent call last):
  File "/Users/gustavo/anaconda3/lib/python3.7/multiprocessing/queues.py", line 242, in _feed
    send_bytes(obj)
  File "/Users/gustavo/anaconda3/lib/python3.7/multiprocessing/connection.py", line 200, in send_bytes
    self._send_bytes(m[offset:offset + size])
Traceback (most recent call last):
  File "/Users/gustavo/anaconda3/lib/python3.7/multiprocessing/qu

KeyboardInterrupt: 